# Notebook title

description here

### Import Libraries

In [1]:
import time
import pandas as pd
from nba_api.stats.endpoints import leaguegamelog, boxscoretraditionalv3

pd.set_option('display.max_columns', None)

SEASON = "2025-26"


### Establish Key Functions



In [14]:
######### BOX STATS FULL GAME #########
def team_box_stats_full_game(years):
    '''
    Fill this out later
    '''
    teams = leaguegamelog.LeagueGameLog(
        season=years,
        season_type_all_star="Regular Season",
        player_or_team_abbreviation="T",
    ).get_data_frames()[0]

    # drop unnecessary columns
    teams = teams.drop(columns=["SEASON_ID", "MIN", "VIDEO_AVAILABLE"])
    return teams


######### ROLLING AVG BOX STATS FULL GAME #########
def rolling_avg_team_box_score_stats(years, prev_games=3):
    '''
    Fill this out later
    '''
    teams = leaguegamelog.LeagueGameLog(
        season=years,
        season_type_all_star="Regular Season",
        player_or_team_abbreviation="T",
    ).get_data_frames()[0]

    stats = [
        "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT", 
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PTS", "PLUS_MINUS"
    ]

    teams[stats] = (teams.groupby("TEAM_ID")[stats].transform(lambda s: s.rolling(prev_games).mean().round(2)))
    teams = teams.rename(columns={s: f"{prev_games}_game_roll_avg_{s}" for s in stats})
    teams = teams.drop(columns=["MIN", "VIDEO_AVAILABLE"])

    return teams.dropna(subset=[f"{prev_games}_game_roll_avg_{s}" for s in stats])


######### BOX STATS BY QUARTER #########
def team_box_stats_by_quarter(game_id, start_period, end_period, range_type):
    '''
    Fill this out later
    '''
    box = boxscoretraditionalv3.BoxScoreTraditionalV3(
        game_id=game_id,
        start_period=start_period,
        end_period=end_period, # set 1 for Q1 snapshot, 2 for halftime snapshot, 3 for Q3 snapshot
        range_type=range_type, # use 1 to get data by period (not custom clock range)
        start_range=0,
        end_range=0,
    ).team_stats.get_data_frame()

    # drop unneseccary columns
    box = box.drop(columns=["teamCity", "teamSlug"])

    # convert column to int, total mins played by players on the team (so divide by 5 here)
    box['minutes'] = box['minutes'].astype(str).str.split(':').str[0].astype(int) / 5

    # then find which quarter by dividing by 12
    box['minutes'] = box['minutes'] / 12

    # rename columns for clarity
    box = box.rename(columns={
        "gameId": "GAME_ID",
        "teamId": "TEAM_ID",
        "teamTricode": "TEAM_ABBREVIATION",
        "minutes": "END_Q",
        "teamName": "TEAM_NAME",
        "fieldGoalsMade": "FGM",
        "fieldGoalsAttempted": "FGA",
        "fieldGoalsPercentage": "FG_PCT",
        "threePointersMade": "FG3M",
        "threePointersAttempted": "FG3A",
        "threePointersPercentage": "FG3_PCT",
        "freeThrowsMade": "FTM",
        "freeThrowsAttempted": "FTA",
        "freeThrowsPercentage": "FT_PCT",
        "reboundsOffensive": "OREB",
        "reboundsDefensive": "DREB",
        "reboundsTotal": "REB",
        "assists": "AST",
        "steals": "STL",
        "blocks": "BLK",
        "turnovers": "TOV",
        "foulsPersonal": "PF",
        "points": "PTS",
        "plusMinusPoints": "PLUS_MINUS"
    })
    return box


######### PULL IN-GAME BOX STATS #########
def pull_in_game_box_stats(list_of_game_ids, start_period, end_period, range_type):
    '''
    fill this out later
    '''
    frames = []
    for game_id in list_of_game_ids:
        frame = team_box_stats_by_quarter(
            game_id, 
            start_period=start_period, 
            end_period=end_period, 
            range_type=range_type
        )

        # to avoid getting rate limited by nba api use sleep
        time.sleep(0.3)

        frames.append(frame)

        # un-hash to see game by game completion of this step
        #print(f'{game_id} done')

    df = pd.concat(frames, ignore_index=True)
    return df


######### MERGE IN-GAME BOX STATS WITH WL/DATE INFO #########
def merge_in_game_stats(df1, df2):
    '''
    fill this out later
    '''
    # use merge on left join (finish this later)
    df = df2.merge(
        df1[['GAME_ID', 'TEAM_ID', 'GAME_DATE', 'MATCHUP', 'WL']],
        on=['GAME_ID', 'TEAM_ID'],
        how='left'
    )

    # re-order columns for clarity
    df = df[[
        "GAME_ID", "GAME_DATE", 
        "TEAM_ID", "TEAM_NAME", "TEAM_ABBREVIATION", "MATCHUP", "END_Q",
        "FGM", "FGA", "FG_PCT", "FG3M", "FG3A", "FG3_PCT", "FTM", "FTA", "FT_PCT",
        "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "PF", "PLUS_MINUS", "WL"        
    ]]
    return df



### Pull Full Game Box Score Stats

fill out later

In [4]:
test = team_box_stats_full_game(SEASON)
test.to_csv("../data/team-box-stats(full-game).csv", index=False)
test.head(4)



,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS
0,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,43,97,0.443,11,39,0.282,27,31,0.871,16,36,52,23,6,5,25,26,124,-1
1,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,42,77,0.545,8,32,0.250,17,28,0.607,7,32,39,23,7,2,20,21,109,-10
2,1610612744,GSW,Golden State Warriors,0022500002,2025-10-21,GSW @ LAL,W,38,78,0.487,17,40,0.425,26,29,0.897,9,31,40,29,10,4,19,27,119,10
3,1610612760,OKC,Oklahoma City Thunder,0022500001,2025-10-21,OKC vs. HOU,W,46,104,0.442,13,52,0.250,20,25,0.800,11,27,38,29,12,4,12,27,125,1


### Get Rolling Averages Data

fill this out later

In [23]:
test1 = rolling_avg_team_box_score_stats(SEASON, prev_games=3)
test1.head(4)


,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,3_game_roll_avg_FGM,3_game_roll_avg_FGA,3_game_roll_avg_FG_PCT,3_game_roll_avg_FG3M,3_game_roll_avg_FG3A,3_game_roll_avg_FG3_PCT,3_game_roll_avg_FTM,3_game_roll_avg_FTA,3_game_roll_avg_FT_PCT,3_game_roll_avg_OREB,3_game_roll_avg_DREB,3_game_roll_avg_REB,3_game_roll_avg_AST,3_game_roll_avg_STL,3_game_roll_avg_BLK,3_game_roll_avg_TOV,3_game_roll_avg_PF,3_game_roll_avg_PTS,3_game_roll_avg_PLUS_MINUS
46,22025,1610612744,GSW,Golden State Warriors,0022500097,2025-10-24,GSW @ POR,L,41.00,85.67,0.48,17.00,41.67,0.41,26.00,31.00,0.83,10.33,31.00,41.33,28.67,11.00,3.33,17.67,21.67,125.00,-1.33
56,22025,1610612753,ORL,Orlando Magic,0022500100,2025-10-25,ORL vs. CHI,L,37.67,85.00,0.44,8.00,28.33,0.27,26.67,35.00,0.77,12.00,36.00,48.00,18.67,7.67,5.00,19.00,23.67,110.00,-4.00
59,22025,1610612737,ATL,Atlanta Hawks,0022500101,2025-10-25,ATL vs. OKC,L,38.00,86.67,0.44,11.33,34.33,0.32,22.33,28.67,0.76,10.67,29.33,40.00,25.33,7.67,5.33,16.33,22.67,109.67,-11.00
62,22025,1610612756,PHX,Phoenix Suns,0022500104,2025-10-25,PHX @ DEN,L,40.00,91.00,0.44,14.33,43.67,0.32,16.67,21.67,0.78,15.00,27.00,42.00,23.33,8.67,5.00,18.00,26.33,111.00,-15.00


In [ ]:
test1.to_csv("../data/roll-avg-only(3GAME).csv", index=False)

test2 = rolling_avg_team_box_score_stats(SEASON, prev_games=5)
test2.to_csv("../data/roll-avg-only(5GAME).csv", index=False)

test3 = rolling_avg_team_box_score_stats(SEASON, prev_games=10)
test3.to_csv("../data/roll-avg-only(10GAME).csv", index=False)

test4 = rolling_avg_team_box_score_stats(SEASON, prev_games=15)
test4.to_csv("../data/roll-avg-only(15GAME).csv", index=False)

### Pull In-Game Box Stats Data

Fill this out later

In [22]:
all_ids = test["GAME_ID"].unique().tolist()
len(all_ids)


1230

In [ ]:
##### Pull for end of Q1 (takes about 20min)
q1_df = pull_in_game_box_stats(all_ids, start_period=1, end_period=1, range_type=1)
q1_df_final = merge_in_game_stats(test, q1_df)
q1_df_final.head(4)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,MATCHUP,END_Q,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WL
0,0022500001,2025-10-21,1610612760,Thunder,OKC,OKC vs. HOU,1.0,10,19,0.526,1,7,0.143,6,7,0.857,2,6,8,7,1,1,5,5,-3.0,W
1,0022500001,2025-10-21,1610612745,Rockets,HOU,HOU @ OKC,1.0,12,23,0.522,2,8,0.250,4,4,1.000,3,6,9,7,5,0,3,5,3.0,L
2,0022500002,2025-10-21,1610612747,Lakers,LAL,LAL vs. GSW,1.0,8,19,0.421,1,9,0.111,5,7,0.714,1,9,10,4,2,1,8,5,-6.0,L
3,0022500002,2025-10-21,1610612744,Warriors,GSW,GSW @ LAL,1.0,8,20,0.400,5,10,0.500,7,7,1.000,1,9,10,6,7,1,6,6,6.0,W


In [ ]:
##### Pull for end of Q2 (takes about 20min)
q2_df = pull_in_game_box_stats(all_ids, start_period=1, end_period=2, range_type=1)
q2_df_final = merge_in_game_stats(test, q2_df)
q2_df_final.head(4)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,MATCHUP,END_Q,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WL
0,0022500001,2025-10-21,1610612760,Thunder,OKC,OKC vs. HOU,2.0,19,43,0.442,5,20,0.250,8,9,0.889,5,11,16,14,3,2,7,10,-6.0,W
1,0022500001,2025-10-21,1610612745,Rockets,HOU,HOU @ OKC,2.0,20,43,0.465,7,19,0.368,10,10,1.000,7,15,22,13,5,1,8,10,6.0,L
2,0022500002,2025-10-21,1610612747,Lakers,LAL,LAL vs. GSW,2.0,20,36,0.556,5,17,0.294,9,16,0.563,3,18,21,10,4,2,14,13,-1.0,L
3,0022500002,2025-10-21,1610612744,Warriors,GSW,GSW @ LAL,2.0,16,40,0.400,7,18,0.389,16,18,0.889,6,15,21,13,8,1,13,14,1.0,W


In [ ]:
##### Pull for end of Q3 (takes about 20min)
q3_df = pull_in_game_box_stats(all_ids, start_period=1, end_period=3, range_type=1)
q3_df_final = merge_in_game_stats(test, q3_df)
q3_df_final.head(4)


,GAME_ID,GAME_DATE,TEAM_ID,TEAM_NAME,TEAM_ABBREVIATION,MATCHUP,END_Q,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,WL
0,0022500001,2025-10-21,1610612760,Thunder,OKC,OKC vs. HOU,3.0,29,67,0.433,8,33,0.242,9,11,0.818,7,16,23,19,8,2,9,16,-4.0,W
1,0022500001,2025-10-21,1610612745,Rockets,HOU,HOU @ OKC,3.0,27,63,0.429,8,28,0.286,17,17,1.000,13,24,37,15,6,2,15,13,4.0,L
2,0022500002,2025-10-21,1610612747,Lakers,LAL,LAL vs. GSW,3.0,30,56,0.536,6,24,0.250,13,23,0.565,5,24,29,15,5,2,17,17,-11.0,L
3,0022500002,2025-10-21,1610612744,Warriors,GSW,GSW @ LAL,3.0,28,60,0.467,12,28,0.429,22,24,0.917,8,25,33,22,9,1,15,20,11.0,W


In [29]:
q1_df_final.to_csv("../data/in-game-box-stats(EndQ1).csv", index=False)
q2_df_final.to_csv("../data/in-game-box-stats(EndQ2).csv", index=False)
q3_df_final.to_csv("../data/in-game-box-stats(EndQ3).csv", index=False)
